In [3]:
import gurobipy as gp
from gurobipy import GRB
import os
import pandas as pd
directorio=r"C:\Users\melco\OneDrive\Escritorio\pasantia_imple\datos_16"
d={}
B_num={}
p={1:869.434,2:1533.228,3:2430.901,4:3169.766,5:3947.629,6:4690.218,7:5514.84,8:6051.184,9:6693.546,
   10:7264.281,11:8095.383,12:8530.091,13:9088.648,14:9600.03,15:10486.354,16:11616.937,17:12249.156,
   18:12605.908,19:14312.13,20:15063.13,21:16036.058,22:16509.946,23:17643.719,24:18142.476,25:18511.465,
   26:19328.821,27:19802.709,28:20775.637,29:21526.637,30:23232.859,31:23589.611,32:24221.83,33:25352.413,
   34:26238.737,35:26750.119,36:27308.676,37:27743.384,38:28574.486,39:29145.221,40:29787.583,41:30323.927,
   42:31148.549,43:31891.138,44:32669,45:33407.865,46:34305.539,47:34969.332,48:35838.766}
idx_p=gp.tuplelist(p)
maxi=0


for j, archivo in enumerate(os.listdir(directorio)):
    if archivo.endswith('.csv'):
        cont=0
        ruta_archivo = os.path.join(directorio, archivo)
        df = pd.read_csv(ruta_archivo)
        df['Distance'] = df['Distance'].fillna(0)
        df['dist_Acum'] = df['Distance'].cumsum()
        filtered_df = df[df['Speed[m/s]']<=3.03]
        aux=df['dist_Acum'].iloc[-1]
        if aux>=maxi:
            maxi=aux
        k=0
        for i, row in filtered_df.iterrows():
            k=k+1
            d[(j+1, k)] = row['dist_Acum']
            cont=cont+1
            B_num[j+1]=cont
        #print(j)

B = {}

M=maxi
# Iterar sobre el diccionario original
for clave, valor in d.items():
    primer_valor = clave[0]
    if primer_valor not in B:
        B[primer_valor] = {}
    B[primer_valor][clave] = valor
    
B_max=max(list(B_num.values()))
print(B_max)

print(M)
idx_d=gp.tuplelist(d)
#print(d)
#print(p)
#print(B[1])
print(B_num)
idx_B=gp.tuplelist(B_num)
print(idx_B)
#print(idx_d)
S=sum(B_num.values())
print(S)

1200
39223.01375818591
{1: 951, 2: 1098, 3: 870, 4: 830, 5: 824, 6: 861, 7: 842, 8: 1200, 9: 926, 10: 797, 11: 1183, 12: 672, 13: 1149, 14: 1182, 15: 670, 16: 803}
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
14858


In [7]:
########### 1 FASE #############
m = gp.Model()
x=m.addVars(idx_d,idx_p, vtype = GRB.BINARY, name="x")
y=m.addVars(idx_B, vtype=GRB.CONTINUOUS, name="y")
z=m.addVars(idx_d,vtype=GRB.CONTINUOUS, name="z", ub=300)
alpha=m.addVar(vtype=GRB.CONTINUOUS, name="alpha",ub=1)
m.setObjective(alpha, GRB.MAXIMIZE)#-x.sum("*","*","*")
S=sum(B_num.values())
m.addConstrs((x.sum(i,"*","*")>=48*alpha for i in idx_B),"max_base")
m.addConstrs((x.sum(i,j,"*")<=1 for (i,j) in idx_d ),"max_punto") #for i in idx_B for j in range(1,B_max+1)
m.addConstrs((z[i,j]>=d[i,j]+y[i]-p[k]-M*(1-x[i,j,k]) for (i,j) in idx_d for k in idx_p),"cota_inf")
m.addConstrs((z[i,j]>=-d[i,j]-y[i]+p[k]-M*(1-x[i,j,k]) for (i,j) in idx_d for k in idx_p),"cota_sup")
#m.addConstr(x.sum("*","*","*")>=(0.1*S),"num_datos")
m.addConstrs((x.sum(i,"*",k)<=1 for i in idx_B for k in idx_p),"unicidad")
m.Params.TimeLimit = 1000
m.Params.NoRelHeurTime = 600
m.optimize()

Set parameter TimeLimit to value 1000
Set parameter NoRelHeurTime to value 600
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11.0 (22631.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-1255U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 1442010 rows, 728059 columns and 6418672 nonzeros
Model fingerprint: 0x892910fe
Variable types: 14875 continuous, 713184 integer (713184 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+04]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 3e+02]
  RHS range        [1e+00, 8e+04]
Found heuristic solution: objective -0.0000000
Presolve removed 386846 rows and 14858 columns (presolve time = 11s) ...
Presolve removed 1326834 rows and 666518 columns
Presolve time: 11.80s
Presolved: 115176 rows, 61541 columns, 385753 nonzeros
Variable types: 17 continuous, 61524 integer (61524 binary)
Starting NoRel heuristic
Found heuristic solu

In [9]:
############## 2 FASE #################
m = gp.Model()
x=m.addVars(idx_d,idx_p, vtype = GRB.BINARY, name="x")
y=m.addVars(idx_B, vtype=GRB.CONTINUOUS, name="y")
z=m.addVars(idx_d,vtype=GRB.CONTINUOUS, name="z", ub=300)
m.setObjective(z.sum("*","*"), GRB.MINIMIZE)#-x.sum("*","*","*")
S=sum(B_num.values())
m.addConstrs((x.sum(i,"*","*")>=48*0.85 for i in idx_B),"max_base")
m.addConstrs((x.sum(i,j,"*")<=1 for (i,j) in idx_d ),"max_punto") #for i in idx_B for j in range(1,B_max+1)
m.addConstrs((z[i,j]>=d[i,j]+y[i]-p[k]-M*(1-x[i,j,k]) for (i,j) in idx_d for k in idx_p),"cota_inf")
m.addConstrs((z[i,j]>=-d[i,j]-y[i]+p[k]-M*(1-x[i,j,k]) for (i,j) in idx_d for k in idx_p),"cota_sup")
#m.addConstr(x.sum("*","*","*")>=(0.1*S),"num_datos")
m.addConstrs((x.sum(i,"*",k)<=1 for i in idx_B for k in idx_p),"unicidad")
m.Params.TimeLimit = 1000
m.Params.NoRelHeurTime = 600
m.optimize()

Set parameter TimeLimit to value 1000
Set parameter NoRelHeurTime to value 600
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11.0 (22631.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-1255U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 1442010 rows, 728058 columns and 6418656 nonzeros
Model fingerprint: 0xceaf00d6
Variable types: 14874 continuous, 713184 integer (713184 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+04]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 3e+02]
  RHS range        [1e+00, 8e+04]
Presolve removed 380758 rows and 0 columns (presolve time = 10s) ...
Presolve removed 1315172 rows and 652124 columns
Presolve time: 10.81s
Presolved: 126838 rows, 75934 columns, 521340 nonzeros
Variable types: 14410 continuous, 61524 integer (61524 binary)
Starting NoRel heuristic
Found phase-1 solution: relaxation 191618
Found phase-1 solution: re

In [11]:
m.write("test.lp")
if m.SolCount > 0:
    # Recuperar los valores de las variables
    vx = m.getAttr('x', x)
    vy = m.getAttr('x', y) 
    vz = m.getAttr('x', z)
    for k in idx_p:
        #print('\nFlujos optimos para {}:'.format(k))
        for i,j in idx_d:
            if vx[i,j,k] > 0:
                print("Base de datos {} punto elegido {} para la parada {}.".format(i, j,k))
    for i in idx_B:
        if vy[i]!=0:
             print(f"Desplazamiento de {i} : {vy[i]}") 
    for i,j in idx_d:
        if vz[i,j]!=0:
            print(f"El error del punto {j} con valor {d[i,j]}en la base {i} es {vz[i,j]}")

Base de datos 1 punto elegido 16 para la parada 1.
Base de datos 2 punto elegido 13 para la parada 1.
Base de datos 3 punto elegido 3 para la parada 1.
Base de datos 4 punto elegido 11 para la parada 1.
Base de datos 5 punto elegido 24 para la parada 1.
Base de datos 6 punto elegido 1 para la parada 1.
Base de datos 8 punto elegido 10 para la parada 1.
Base de datos 12 punto elegido 2 para la parada 1.
Base de datos 13 punto elegido 28 para la parada 1.
Base de datos 15 punto elegido 19 para la parada 1.
Base de datos 16 punto elegido 14 para la parada 1.
Base de datos 1 punto elegido 27 para la parada 2.
Base de datos 2 punto elegido 30 para la parada 2.
Base de datos 3 punto elegido 13 para la parada 2.
Base de datos 4 punto elegido 13 para la parada 2.
Base de datos 5 punto elegido 49 para la parada 2.
Base de datos 6 punto elegido 8 para la parada 2.
Base de datos 7 punto elegido 20 para la parada 2.
Base de datos 8 punto elegido 23 para la parada 2.
Base de datos 10 punto elegido 

In [13]:
m.Params.TimeLimit = 7200
m.Params.NoRelHeurTime = 5000
m.reset()
m.optimize()

Set parameter TimeLimit to value 7200
Set parameter NoRelHeurTime to value 5000
Discarded solution information
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11.0 (22631.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-1255U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 1442010 rows, 728058 columns and 6418656 nonzeros
Model fingerprint: 0xceaf00d6
Variable types: 14874 continuous, 713184 integer (713184 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+04]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 3e+02]
  RHS range        [1e+00, 8e+04]
Presolve removed 380758 rows and 0 columns (presolve time = 10s) ...
Presolve removed 1315172 rows and 652124 columns
Presolve time: 10.99s
Presolved: 126838 rows, 75934 columns, 521340 nonzeros
Variable types: 14410 continuous, 61524 integer (61524 binary)
Starting NoRel heuristic
Found phase-1 solution: relaxation 1

In [15]:
m.Params.TimeLimit = 19000
m.Params.NoRelHeurTime = 18000
m.reset()
m.optimize()

Set parameter TimeLimit to value 19000
Set parameter NoRelHeurTime to value 18000
Discarded solution information
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11.0 (22631.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-1255U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 1442010 rows, 728058 columns and 6418656 nonzeros
Model fingerprint: 0xceaf00d6
Variable types: 14874 continuous, 713184 integer (713184 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+04]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 3e+02]
  RHS range        [1e+00, 8e+04]
Presolve removed 380758 rows and 0 columns (presolve time = 11s) ...
Presolve removed 1315172 rows and 652124 columns
Presolve time: 11.90s
Presolved: 126838 rows, 75934 columns, 521340 nonzeros
Variable types: 14410 continuous, 61524 integer (61524 binary)
Starting NoRel heuristic
Found phase-1 solution: relaxation

In [17]:
m.write("modelo_16 bases_50 paradas.lp")
if m.SolCount > 0:
    # Recuperar los valores de las variables
    vx = m.getAttr('x', x)
    vy = m.getAttr('x', y) 
    vz = m.getAttr('x', z)
    for k in idx_p:
        #print('\nFlujos optimos para {}:'.format(k))
        for i,j in idx_d:
            if vx[i,j,k] > 0:
                print("Base de datos {} punto elegido {} para la parada {}.".format(i, j,k))
    for i in idx_B:
        if vy[i]!=0:
             print(f"Desplazamiento de {i} : {vy[i]}") 
    for i,j in idx_d:
        if vz[i,j]!=0:
            print(f"El error del punto {j} con valor {d[i,j]}en la base {i} es {vz[i,j]}")

Base de datos 1 punto elegido 16 para la parada 1.
Base de datos 2 punto elegido 13 para la parada 1.
Base de datos 3 punto elegido 3 para la parada 1.
Base de datos 4 punto elegido 11 para la parada 1.
Base de datos 5 punto elegido 24 para la parada 1.
Base de datos 6 punto elegido 1 para la parada 1.
Base de datos 8 punto elegido 10 para la parada 1.
Base de datos 9 punto elegido 14 para la parada 1.
Base de datos 11 punto elegido 4 para la parada 1.
Base de datos 12 punto elegido 2 para la parada 1.
Base de datos 13 punto elegido 28 para la parada 1.
Base de datos 15 punto elegido 19 para la parada 1.
Base de datos 16 punto elegido 10 para la parada 1.
Base de datos 1 punto elegido 27 para la parada 2.
Base de datos 2 punto elegido 30 para la parada 2.
Base de datos 3 punto elegido 13 para la parada 2.
Base de datos 4 punto elegido 13 para la parada 2.
Base de datos 5 punto elegido 49 para la parada 2.
Base de datos 6 punto elegido 8 para la parada 2.
Base de datos 7 punto elegido 2

In [ ]:
base1=[663.9889299999999,1300.2893700000002,0,2959.477629999997,3668.2424499999984,4287.68361,0,5771.377949999995,6319.318529999995,6919.098549999995,7679.125839999994,8405.229279999992,8879.545029999994,9448.762839999996,10003.77562,11438.032580000006,11879.261890000002,12174.192740000006,13955.737510000003,]
base2=[]